# Transfer learning and imaging foundation models

Exercise notebook for the lesson **[Transfer learning and imaging foundation models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models)** in the *Deep learning & imaging* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Transfer learning reuses a model trained on a large dataset as the starting point for a new task, so far fewer labelled images are needed. You can fine-tune all of its layers on the new data or freeze the learned features and retrain only the final layer. Imaging foundation models take this further: pretrained on millions of unlabelled medical images, such as retinal photographs, they can be adapted to many tasks with little labelled data. A pretrained model still has to be validated at each new site, because used as it is, it can fail on a different scanner.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[CNNs for chest X-ray and retinal images](https://ramkumarmd.com/medical-ai/deep-learning-imaging/cnns-medical-imaging)**: Synthetic scans, a two-layer CNN and its test accuracy. Takes a few seconds on a CPU.

In [ ]:
import numpy as np
import torch
from torch import nn

def make_images(n, seed, marker_with_lesion=0.0):
    """Synthetic 28x28 'scans': noisy background; half contain a faint round lesion at a random place.
    marker_with_lesion: chance that lesion images also carry a bright corner marker (a shortcut)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:28, :28]
    images = rng.normal(0, 1, (n, 28, 28))
    labels = rng.integers(0, 2, n)
    for i in np.where(labels == 1)[0]:
        cy, cx, r = rng.integers(6, 22), rng.integers(6, 22), rng.uniform(2.5, 4.0)
        images[i] += 1.0 * ((yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2)
        if rng.random() < marker_with_lesion:
            images[i, :3, :3] += 4.0                     # e.g. a portable-scanner label
    return torch.tensor(images, dtype=torch.float32).unsqueeze(1), torch.tensor(labels)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 28 -> 14
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
    def forward(self, x):
        return self.head(self.features(x))

def train(model, X, y, epochs=8, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), y[idx]).backward()
            opt.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

X_train, y_train = make_images(4000, seed=1)
X_test, y_test = make_images(1000, seed=2)
torch.manual_seed(0)
cnn = train(SmallCNN(), X_train, y_train)
print(f"CNN test accuracy: {accuracy(cnn, X_test, y_test):.3f}")
print(f"Parameters: {sum(p.numel() for p in cnn.parameters()):,}")

# CNN test accuracy: 0.946
# Parameters: 2,818

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Transfer to a new scanner with 100 images

Run the first block of the CNN lesson first. Compare four ways of getting a model for site B.

In [ ]:
import copy

def new_scanner(n, seed):
    """Target site: a different scanner gives lower contrast and an intensity offset."""
    X, y = make_images(n, seed)
    return 0.6 * X + 0.5, y

X_source, y_source = make_images(4000, seed=10)          # plenty of labelled images from site A
X_small, y_small = new_scanner(100, seed=11)              # only 100 labelled images at site B
X_target_test, y_target_test = new_scanner(1000, seed=12)

torch.manual_seed(0)
pretrained = train(SmallCNN(), X_source, y_source)
print(f"Pretrained model, applied to site B as is:  {accuracy(pretrained, X_target_test, y_target_test):.3f}")

torch.manual_seed(0)
scratch = train(SmallCNN(), X_small, y_small, epochs=30)
print(f"Trained from scratch on 100 site-B images:  {accuracy(scratch, X_target_test, y_target_test):.3f}")

fine_tuned = train(copy.deepcopy(pretrained), X_small, y_small, epochs=30)
print(f"Pretrained, fine-tuned on 100 site-B images: {accuracy(fine_tuned, X_target_test, y_target_test):.3f}")

# Pretrained model, applied to site B as is:  0.504
# Trained from scratch on 100 site-B images:  0.724
# Pretrained, fine-tuned on 100 site-B images: 0.854

### Freezing the learned features

Continuing from the previous block: keep the convolutional filters fixed and retrain only the final layer.

In [ ]:
frozen = copy.deepcopy(pretrained)
for p in frozen.features.parameters():
    p.requires_grad = False                       # keep the learned filters; retrain only the head
frozen.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
torch.manual_seed(0)
opt = torch.optim.Adam(frozen.head.parameters(), lr=1e-3)
for _ in range(30):
    for idx in torch.randperm(len(X_small)).split(64):
        opt.zero_grad()
        nn.functional.cross_entropy(frozen(X_small[idx]), y_small[idx]).backward()
        opt.step()
print(f"Frozen features, new head on 100 images:    {accuracy(frozen, X_target_test, y_target_test):.3f}")
trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable} of {sum(p.numel() for p in frozen.parameters())}")

# Frozen features, new head on 100 images:    0.772
# Trainable parameters: 1570 of 2818

## Exercise

**Goal:** Compare transfer-learning strategies on synthetic and real images, and plan the validation of a pretrained model.

**Data:** Synthetic scans in the notebook; optional MedMNIST RetinaMNIST or PneumoniaMNIST; optional torchvision pretrained ResNet-18 (Generated in the notebook; MedMNIST under CC BY 4.0; check the licence of any pretrained weights)

### Step 1

Reproduce the four strategies and repeat with 25, 100 and 500 site-B images; plot accuracy against the number of labelled images.

In [ ]:
# Your code here


### Step 2

Normalise each site's images (subtract the mean, divide by the standard deviation) and see whether the pretrained model works at site B without fine-tuning.

In [ ]:
# Your code here


### Step 3

Optional: fine-tune an ImageNet-pretrained ResNet-18 on RetinaMNIST and compare with training the same network from scratch.

In [ ]:
# Your code here


### Step 4

Write the validation plan you would require before using an imaging foundation model in a diabetic retinopathy screening programme.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.